# UABR-SLM · Notebook 04 (v2) — G1 pilot after fact re-injection

**Phase 04, version 2** (PLAN §11 G1, §16). GPU **T4** required · ~25–40 min.

**What changed from v1 (10 Nov 2026):**
- v1 probed with cloze + 3 demonstrations → the 40M model **copied demonstration answers** (59 % exact copies;
  one answer per relation for ≥ 94 % of questions). v2 uses **zero-shot cloze** as the generation probe.
- Generation now stops only at a newline; answer cleaning no longer cuts names at initials ("John H. Smith").
- Two probes that do not depend on free generation:
  **ranking** (gold vs 30 same-relation distractors by log-prob; with vs without the subject) and
  **verbatim** (prefix of an injected fact sentence → does the model continue with the answer?).
- Both models are probed: **base** (notebook 03) and **re-injected** (notebook 03b). The randomized **doses
  0 / 16 / 64 / 256** replace the old exposure groups. The base model doubles as a randomization check
  (it should show no dose effect).

**G1 criteria (v2):**
| check | pass condition (re-injected model) |
|---|---|
| G1a knowledge beyond the relation prior | pilot zero-shot closed-book EM − relation-prior EM ≥ 5 pp, 95 % CI lower bound > 0 |
| G1b knowledge rises with dose | EM **or** ranking accuracy@1: dose 256 − dose 0 difference has CI lower bound > 0 (in-main val) |
| G1c b not almost all positive | P(b_quality > 0) < 90 % on the pilot |
| G1d retrieval damage exists | ≥ 1 pilot question correct closed-book but wrong with evidence |

**Inputs:** outputs of notebooks **00, 01, 02, 03, 03b**. **Kaggle:** Accelerator **GPU T4** · Internet **On** · Save & Run All.

## 1 · Get the project code

In [ ]:
import os, sys, glob, json, subprocess, platform, time, shutil
ON_KAGGLE = os.path.exists("/kaggle/working")

REPO_URL = "https://github.com/Prajwal-13-dev/uabr_slm.git"
REPO_REF = "main"        # pin to a commit hash for final, reportable runs

def find_code_root():
    if os.environ.get("UABR_CODE"):
        return os.path.abspath(os.environ["UABR_CODE"])
    for base in glob.glob("/kaggle/input/*"):
        for root, dirs, _ in os.walk(base):
            if os.path.isdir(os.path.join(root, "src", "uabr")):
                return root
            if root.count(os.sep) - base.count(os.sep) >= 4:
                dirs[:] = []
    dest = "/tmp/uabr_slm"
    if not os.path.isdir(dest):
        url = REPO_URL
        try:
            from kaggle_secrets import UserSecretsClient
            tok = UserSecretsClient().get_secret("GITHUB_TOKEN")
            url = url.replace("https://", f"https://{tok}@")
        except Exception:
            pass
        subprocess.run(["git", "clone", "-q", url, dest], check=True)
        subprocess.run(["git", "-C", dest, "checkout", "-q", REPO_REF], check=True)
    return dest

CODE_ROOT = find_code_root()
SRC = os.path.join(CODE_ROOT, "src")
sys.path.insert(0, SRC)
CODE_COMMIT = subprocess.run(["git", "-C", CODE_ROOT, "rev-parse", "HEAD"],
                             capture_output=True, text=True).stdout.strip() or "unknown"

def find_input(pattern, env=None):
    """Locate a file from an attached notebook output (or an env override for local tests)."""
    if env and os.environ.get(env):
        return os.environ[env]
    hits = sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))
    assert hits, f"{pattern} not found under /kaggle/input - attach the required notebook output (Add Input → Your Work)"
    return hits[0]

T0 = time.time()
def stage(msg): print(f"[{(time.time()-T0)/60:6.1f} min] {msg}", flush=True)
print("code:", CODE_ROOT, "| commit:", CODE_COMMIT[:10])

import torch
from uabr.data import tokenizer as T
from uabr.eval import prompts as PR
from uabr.eval.generate import load_model, encode, fit_evidence, generate, answer_mean_logprob, seed_for
from uabr.eval.metrics import alias_em, alias_f1, alias_contains, normalize_answer
from uabr.eval.stats import clustered_bootstrap_mean, clustered_bootstrap_diff
from uabr.eval.probes import score_continuations, candidate_sets, answer_span
from uabr.retrieval.select import select_non_overlapping

## 2 · Configuration and inputs

In [ ]:
import numpy as np, pandas as pd
OUT = "/kaggle/working/04_pilot_v2" if ON_KAGGLE else os.path.abspath("outputs/04_pilot_v2")
os.makedirs(OUT, exist_ok=True)
DEV = "cuda" if torch.cuda.is_available() else "cpu"
if ON_KAGGLE: assert DEV == "cuda", "No GPU - set Accelerator to GPU T4 (v1 ran on CPU for 5 h)"

PROMPT_MODE = "zero_shot_cloze"
K, TEMP, MAX_NEW, CTX, SEED = 8, 0.7, 16, 512, 2026
N_DISTRACT = 30
MIN_GAIN_PP = 5.0
PILOT_LIMIT = int(os.environ.get("PILOT_LIMIT", 0))
VAL_LIMIT = int(os.environ.get("VAL_LIMIT", 0))
VERBATIM_PER_DOSE = int(os.environ.get("VERBATIM_PER_DOSE", 250))
DOSES = ["d0", "d16", "d64", "d256"]

popqa = pd.read_json(find_input("00_popqa_splits/popqa_prepared.jsonl", "POPQA_PREPARED"), lines=True)
tok = T.load(find_input("01_corpus/tokenizer/tokenizer.json", "TOKENIZER"))
top = pd.read_parquet(find_input("02_bm25/retrieval/popqa_top20.parquet", "TOP20"))
wins = pd.read_parquet(find_input("02_bm25/retrieval/windows_retrieved.parquet", "WINDOWS"))
facts = pd.read_parquet(find_input("03b_reinject/facts_doses.parquet", "FACTS"))
MODEL_PATHS = {"base": find_input("03_pretrain/model_final.pt", "MODEL"),
               "reinjected": find_input("03b_reinject/model_final.pt", "MODEL_REINJ")}
MODELS = {k: load_model(p, DEV) for k, p in MODEL_PATHS.items()}

WTEXT = dict(zip(wins["doc_id"], wins["text"]))
WART, WSTART = dict(zip(wins["doc_id"], wins["article_idx"])), dict(zip(wins["doc_id"], wins["start_word"]))
TOPDOCS = dict(zip(top["id"], top["doc_ids"]))
def evidence_docs(qid): return select_non_overlapping(TOPDOCS[qid], WART, WSTART, k=3, window_words=60)
popqa = popqa.merge(facts[["id", "dose_group", "dose", "n_sentences"]], on="id", how="left")
popqa["dose_group"] = popqa["dose_group"].fillna("not_in_main")
stop_ids = [i for t, i in tok.get_vocab().items() if "\n" in tok.decode([i])]
DEMOS = PR.ZERO_SHOT
pilot = popqa[popqa["is_pilot"]].reset_index(drop=True)
if PILOT_LIMIT: pilot = pilot.head(PILOT_LIMIT)
val = popqa[(popqa["split"] == "val") & popqa["in_main"]].reset_index(drop=True)
if VAL_LIMIT: val = val.head(VAL_LIMIT)
print(f"pilot {len(pilot)} | val in-main {len(val)} | device {DEV} | newline stop tokens {len(stop_ids)}")
print("val questions by dose group:", val["dose_group"].value_counts().to_dict())

## 3 · Prompt example (zero-shot)

In [ ]:
r0 = pilot.iloc[0]
print("--- closed-book ---\n" + PR.closed_prompt(r0.prop, r0.subj, DEMOS) + " ▌")
print("\n--- evidence ---\n" + PR.evidence_prompt([WTEXT[d] for d in evidence_docs(r0.id)[0]], r0.prop, r0.subj, DEMOS) + " ▌")

## 4 · Pilot on the re-injected model (greedy + 8 samples, both conditions)

In [ ]:
def decode_clean(ids): return PR.clean_generation(tok.decode(ids))
M = MODELS["reinjected"]
rows = []
for i, r in enumerate(pilot.itertuples(index=False)):
    a_ids = encode(tok, PR.closed_prompt(r.prop, r.subj, DEMOS))
    sel, n_skip = evidence_docs(r.id)
    c_ids, n_used = fit_evidence(tok, lambda w: PR.evidence_prompt(w, r.prop, r.subj, DEMOS), [WTEXT[d] for d in sel], CTX - MAX_NEW)
    rec = {"id": r.id, "evidence_doc_ids": sel[:n_used], "n_windows_used": n_used}
    ans_ids = tok.encode(" " + r.answer, add_special_tokens=False).ids
    for cond, ids in [("A", a_ids), ("C", c_ids)]:
        rec[f"greedy_{cond}"] = decode_clean(generate(M, ids, 1, MAX_NEW, 0.0, 0, DEV, stop_ids)[0])
        rec[f"samples_{cond}"] = [decode_clean(s) for s in generate(M, ids, K, MAX_NEW, TEMP, seed_for(SEED, r.id, cond), DEV, stop_ids)]
        rec[f"lp_{cond}"] = answer_mean_logprob(M, ids, ans_ids, DEV)
    rows.append(rec)
    if (i + 1) % 100 == 0: stage(f"  pilot {i+1}/{len(pilot)}")
res = pilot.merge(pd.DataFrame(rows), on="id")
for cond in "AC":
    res[f"em_{cond}"] = [alias_em(p, g) for p, g in zip(res[f"greedy_{cond}"], res["gold_aliases"])]
    res[f"f1_{cond}"] = [alias_f1(p, g) for p, g in zip(res[f"greedy_{cond}"], res["gold_aliases"])]
    res[f"sample_f1_{cond}"] = [float(np.mean([alias_f1(s, g) for s in ss])) for ss, g in zip(res[f"samples_{cond}"], res["gold_aliases"])]
res["b_quality"] = res["sample_f1_C"] - res["sample_f1_A"]
res["b_likelihood"] = np.exp(res["lp_C"]) - np.exp(res["lp_A"])
top1 = res.groupby("prop")["greedy_A"].agg(lambda s: s.value_counts(normalize=True).iloc[0])
print(f"closed-book EM {res.em_A.mean():.1%} | with evidence {res.em_C.mean():.1%} | relation prior {res.prior_correct.mean():.1%}")
print(f"answer diversity: most common answer per relation covers {top1.mean():.0%} of its questions on average (v1: ~90%)")
res[["prop", "question", "greedy_A", "greedy_C", "answer", "dose_group"]].head(15)

## 5 · All in-main validation questions, both models: zero-shot greedy + ranking probe

In [ ]:
answers_by_rel = (popqa[(popqa["split"] == "train") & popqa["in_main"]].groupby("prop")["answer"]
                  .agg(lambda s: sorted(set(s))).to_dict())
CANDS = candidate_sets(answers_by_rel, {q: (a, g) for q, a, g in zip(val["id"], val["answer"], val["gold_aliases"])},
                       dict(zip(val["id"], val["prop"])), N_DISTRACT, seed=SEED)

def rank_of_gold(model, prompt, cands):
    conts = [tok.encode(" " + c, add_special_tokens=False).ids for c in cands]
    s, m = score_continuations(model, encode(tok, prompt), conts, DEV)
    return int((m > m[0]).sum()), int((s > s[0]).sum()), float(m[0])

out = []
for name, model in MODELS.items():
    for i, r in enumerate(val.itertuples(index=False)):
        g = decode_clean(generate(model, encode(tok, PR.closed_prompt(r.prop, r.subj, DEMOS)), 1, MAX_NEW, 0.0, 0, DEV, stop_ids)[0])
        rk_m, rk_s, gold_lp = rank_of_gold(model, PR.cloze(r.prop, r.subj), CANDS[r.id])
        rk0, _, _ = rank_of_gold(model, PR.cloze(r.prop, "the subject"), CANDS[r.id])
        out.append({"model": name, "id": r.id, "greedy": g, "em": alias_em(g, r.gold_aliases),
                    "rank_mean": rk_m, "rank_sum": rk_s, "rank_nosubj": rk0, "gold_lp": gold_lp})
        if (i + 1) % 500 == 0: stage(f"  {name} val {i+1}/{len(val)}")
V = pd.DataFrame(out).merge(val[["id", "prop", "pop_bin", "dose_group", "group_id", "prior_correct", "gold_aliases"]], on="id")
V["acc1"] = (V["rank_mean"] == 0).astype(float); V["acc1_sum"] = (V["rank_sum"] == 0).astype(float)
V["acc1_nosubj"] = (V["rank_nosubj"] == 0).astype(float); V["rr"] = 1 / (1 + V["rank_mean"])
summ = V.groupby("model").agg(EM=("em", "mean"), prior=("prior_correct", "mean"), rank_acc1=("acc1", "mean"),
                             rank_acc1_sum=("acc1_sum", "mean"), rank_acc1_no_subject=("acc1_nosubj", "mean"), MRR=("rr", "mean"))
summ["chance_acc1"] = 1 / (N_DISTRACT + 1)
display(summ.round(3))
for name in MODELS:
    v = V[V.model == name]
    d = clustered_bootstrap_mean((v["acc1"] - v["acc1_nosubj"]) * 100, v["group_id"])
    print(f"{name:10s} ranking knowledge (with - without subject): {d['mean']:+.1f} pp [{d['lo']:+.1f}, {d['hi']:+.1f}]")

## 6 · G1b — dose response (randomized doses; in-main val)

In [ ]:
dose_tabs, D = {}, {}
for name in MODELS:
    v = V[(V.model == name) & V.dose_group.isin(DOSES)]
    dose_tabs[name] = v.groupby("dose_group").agg(n=("id", "size"), EM=("em", "mean"), rank_acc1=("acc1", "mean"),
                                                  gold_lp=("gold_lp", "mean")).reindex(DOSES)
    for metric in ["em", "acc1", "gold_lp"]:
        D[(name, metric)] = clustered_bootstrap_diff(v[metric], v["group_id"], v.dose_group == "d256", v.dose_group == "d0")
    print(f"\n{name}:"); display(dose_tabs[name].round(3))
    for metric in ["em", "acc1", "gold_lp"]:
        d = D[(name, metric)]; sc = 100 if metric != "gold_lp" else 1
        print(f"  d256 - d0 {metric:8s}: {d['diff']*sc:+.2f} [{d['lo']*sc:+.2f}, {d['hi']*sc:+.2f}] {'pp' if sc == 100 else 'nats'}")
print("\npopularity (re-injected, EM / rank acc@1):")
display(V[V.model == "reinjected"].groupby("pop_bin")[["em", "acc1"]].mean().round(3))

## 7 · Verbatim probe — does the model complete its injected fact sentences?

In [ ]:
gold = dict(zip(popqa["id"], popqa["gold_aliases"]))
ver = []
fd = facts[facts.dose_group.isin(DOSES) & (facts.n_sentences > 0)]
for g, sub in fd.groupby("dose_group"):
    for r in sub.sample(min(VERBATIM_PER_DOSE, len(sub)), random_state=5).itertuples(index=False):
        sp = answer_span(r.sentences[0], gold[r.id])
        if sp: ver.append((g, r.id, r.sentences[0][:sp[0]].rstrip(), sp[1]))
vrows = []
for name, model in MODELS.items():
    for g, qid, prefix, ans in ver:
        p_ids = [T.BOS_ID] + tok.encode(prefix, add_special_tokens=False).ids
        cont = tok.decode(generate(model, p_ids, 1, MAX_NEW, 0.0, 0, DEV, stop_ids)[0])
        _, m = score_continuations(model, p_ids, [tok.encode(" " + ans, add_special_tokens=False).ids], DEV)
        vrows.append({"model": name, "dose_group": g, "id": qid, "completes": alias_contains(cont, [ans], min_chars=3), "lp": float(m[0])})
VB = pd.DataFrame(vrows)
vt = VB.pivot_table(index="dose_group", columns="model", values=["completes", "lp"], aggfunc="mean").reindex(DOSES)
display(vt.round(3))

## 8 · G1c / G1d — benefit distribution, 4-way table, retrieval damage (re-injected, pilot)

In [ ]:
b = res["b_quality"]
b_dist = {"P(b>0)": float((b > 0).mean()), "P(b<0)": float((b < 0).mean()), "P(b=0)": float((b == 0).mean()),
          "P(|b|<=0.05)": float((b.abs() <= 0.05).mean()), "mean_b": float(b.mean())}
print({k: round(v, 3) for k, v in b_dist.items()})
four = pd.crosstab(res["em_A"].map({1.0: "closed correct", 0.0: "closed wrong"}), res["em_C"].map({1.0: "evidence correct", 0.0: "evidence wrong"}))
display(four)
damage = int(((res.em_A == 1) & (res.em_C == 0)).sum())
gain = clustered_bootstrap_mean((res["em_A"] - res["prior_correct"]) * 100, res["group_id"])
print(f"G1a pilot gain over prior: {gain['mean']:+.1f} pp [{gain['lo']:+.1f}, {gain['hi']:+.1f}] | retrieval damage: {damage}")
display(res.groupby("dose_group")[["em_A", "em_C", "b_quality"]].mean().round(3))

## 9 · Save

In [ ]:
res.to_parquet(f"{OUT}/pilot_results.parquet", index=False)
V.drop(columns=["gold_aliases"]).to_parquet(f"{OUT}/val_probes.parquet", index=False)
VB.to_parquet(f"{OUT}/verbatim_probe.parquet", index=False)
summ.to_csv(f"{OUT}/val_summary_by_model.csv")
for name, t in dose_tabs.items(): t.to_csv(f"{OUT}/dose_response_{name}.csv")
four.to_csv(f"{OUT}/four_way_table.csv")
json.dump({"notebook": "04_pilot_v2", "prompt_mode": PROMPT_MODE, "templates": PR.CLOZE_TEMPLATES, "k": K, "temperature": TEMP,
           "max_new": MAX_NEW, "n_distract": N_DISTRACT, "models": MODEL_PATHS, "pilot_gain_pp": gain, "b_distribution": b_dist,
           "retrieval_damage": damage, "dose_diffs": {f"{k[0]}:{k[1]}": v for k, v in D.items()},
           "code": {"repo": REPO_URL, "ref": REPO_REF, "commit": CODE_COMMIT}}, open(f"{OUT}/meta.json", "w"), indent=2, default=str)

## 10 · Gate G1 (v2)

In [ ]:
checks = []
def add(name, ok, detail, severity="fail"):
    checks.append({"name": name, "passed": bool(ok), "detail": detail, "severity": severity})
add("G1a_gain_over_prior_ge_5pp", gain["mean"] >= MIN_GAIN_PP and gain["lo"] > 0, f"{gain['mean']:+.1f} pp [CI {gain['lo']:+.1f}, {gain['hi']:+.1f}]")
de, da = D[("reinjected", "em")], D[("reinjected", "acc1")]
add("G1b_rises_with_dose", de["lo"] > 0 or da["lo"] > 0,
    f"d256-d0 EM {de['diff']*100:+.1f} pp (lo {de['lo']*100:+.1f}); rank acc@1 {da['diff']*100:+.1f} pp (lo {da['lo']*100:+.1f})")
add("G1c_b_not_almost_all_positive", b_dist["P(b>0)"] < 0.90, f"P(b>0) = {b_dist['P(b>0)']:.1%}")
add("G1d_retrieval_damage_exists", damage >= 1, f"{damage} questions")
bb = D[("base", "acc1")]
add("info_randomization_check_base_no_dose_effect", bb["lo"] <= 0 <= bb["hi"], f"base d256-d0 rank acc@1 {bb['diff']*100:+.1f} pp [{bb['lo']*100:+.1f}, {bb['hi']*100:+.1f}]", "warn")

fails = [c for c in checks if not c["passed"] and c["severity"] == "fail"]
warns = [c for c in checks if not c["passed"] and c["severity"] == "warn"]
status = "FAIL" if fails else ("PASS_WITH_WARNINGS" if warns else "PASS")
for c in checks:
    print(f"[{'PASS' if c['passed'] else c['severity'].upper():4}] {c['name']:<38} {c['detail']}")

json.dump({"gate": "G1_v2", "status": status, "checks": checks}, open(f"{OUT}/gate_report.json", "w"), indent=2, default=str)
print("\nGATE G1 (v2):", status)

## Next
- **G1 PASS:** notebook 05 (reading preparation) starts from the **re-injected** model.
- **G1 FAIL:** read the dose-response and verbatim tables first: *verbatim rises but ranking does not* = memorised
  but not extractable (wording problem); *nothing rises* = capacity/dose problem → instructor decision (larger model).